In [1]:
import re 

In [11]:
class RegexRouter:
    def __init__(self):
        self.patterns = [
            # Ticket-Year  dashboard
            {
                "name": "dashboard",
                "regex": re.compile(r'\b([A-Z]{1,5})\s+(\d{4})\b'),
                "format": "ROUTE:DASHBOARD|{0}|{1}"
            },
            {
                "name": "exact_metric",
                "regex": re.compile(r'\b([A-Z]{1,5})\s+(?:(?:in|for)?\s*(\d{4})\s+)?([Rr]evenue|[Nn]et\s+[Ii]ncome|[Gg]ross\s+[Mm]argin|[Oo]perating\s+[Mm]argin)(?:\s+(?:in|for)?\s*(\d{4}))?\b'),
                "format": "ROUTE:EXACT_METRIC|{0}|{2}|{1}{3}" # Handles year before or after metric
            },
            {
                "name": "sec_doc",
                "regex": re.compile(r'\b([A-Z]{1,5})\s+(10-K|10-Q|8-K)(?:\s+(\d{4}))?\b', re.IGNORECASE),
                "format": "ROUTE:FETCH_DOC|{0}|{1}|{2}"
            },
            {
                "name": "cik_lookup",
                "regex": re.compile(r'\bCIK\s+(\d{10})\b', re.IGNORECASE),
                "format": "ROUTE:CIK_LOOKUP|{0}"
            }
        ]
        
    def get(self,query:str) -> str | None :
        """
        Evaluates the query against per-compiled regex patterns 
        """
        
        for pattern in self.patterns:
            match = pattern["regex"].search(query)
            if match:
                groups = match.groups()
                
                if pattern["name"] == "exact_metric":
                    ticker = groups[0].upper()
                    metric = groups[2].lower()
                    year = groups[1] if groups[1] else groups[3]
                    return f"ROUTE:EXACT_METRIC|{ticker}|{metric}|{year}"
                
                safe_groups = [g if g else "None" for g in groups]
                if pattern["name"] in ["dashboard","sec_doc"]:
                    safe_groups[0] = safe_groups[0].upper()
                    
                return pattern["format"].format(*safe_groups)
        
        return None
        

In [ ]:
if __name__ == "__main__":
    import time

    print("Initializing Regex Router...")
    router = RegexRouter()
    print("Router ready.\n")

    tests = [
        "Can you show me the AAPL 2023 dashboard?",
        "What was TSLA net income in 2022?",
        "Fetch the META 10-K",
        "Search for CIK 0000320193 please",
        "Why did Apple's revenue drop in Q3?" 
    ]

    for i, test_query in enumerate(tests, 1):
        print(f"Test {i}: '{test_query}'")
        
        # Using nanoseconds for higher precision since regex is incredibly fast
        start = time.perf_counter()
        result = router.get(test_query)
        latency = (time.perf_counter() - start) * 1000
        
        print(f"  -> Result:  {result}")
        print(f"  -> Latency: {latency:.4f} ms\n")

Initializing Regex Router...
Router ready.

Test 1: 'Can you show me the AAPL 2023 dashboard?'
  -> Result:  ROUTE:DASHBOARD|AAPL|2023
  -> Latency: 0.0299 ms

Test 2: 'What was TSLA net income in 2022?'
  -> Result:  ROUTE:EXACT_METRIC|TSLA|net income|2022
  -> Latency: 0.0203 ms

Test 3: 'Fetch the META 10-K'
  -> Result:  ROUTE:FETCH_DOC|META|10-K|None
  -> Latency: 0.0245 ms

Test 4: 'Search for CIK 0000320193 please'
  -> Result:  ROUTE:CIK_LOOKUP|0000320193
  -> Latency: 0.0238 ms

Test 5: 'Why did Apple's revenue drop in Q3?'
  -> Result:  None
  -> Latency: 0.0157 ms

